# 3 · Whole frames, in the client's tiles

Notebooks 1 and 2 score 80 px samples, each cut where something moves. The client sees something
else: every 328×188 tile of every frame, one motion found per tile, a scrolled page beside a still
panel in the same tile, and mostly stillness. Extrapolation's samples flattered its search by half
against its whole tiles (its notebook 4). So here the validation clips are rebuilt whole
(`helpers/clips.py`) and cut as the client cuts them, at every received frame N+2 a live capture
could give (`runs.valid`), and every method is scored over the region the client keeps from each
tile:

- **changed tiles**: N+1 differs from N or from N+2 over the kept region (`runs.MOVING_SHARE` of it
  by more than 8 luma levels) - holding either frame is wrong there;
- **moving tiles**: the published search's gate is open - something moved between N and N+2.
  "Changed" cannot judge a tile the search leaves as it is, so the later notebooks are scored here;
- every tile capped at 50 dB, as extrapolation's later notebooks, so a still tile's near-perfect
  score does not rule a mean;
- the range is 95% of a bootstrap over videos (over clips for the drawn kinds), the unit a
  recording is split by.

**The reach.** The published search covers ±8 stream pixels between N and N+2 - two display
frames of motion. On notebook 1's samples the exact shift, which searches ±16, is 1.4 dB above it
on the changed samples and 5 dB on the pages (38.64 against 33.60). Three searches are scored here
(`decide.SEARCHES`): the published ±8, the same at ±16 (289 offsets for 81), and ±16 at a quarter
of the resolution (81 offsets, 4 px apart). The one the later notebooks build on is chosen on the
training split's moving tiles, among those the client can afford.

The methods: for each search, the six pictures its motion can make (`models.Decided.CHOICES`) - the
search, the mock's mean, either frame held, either frame moved alone - and notebook 1's per-pixel
pick on the published search. Nothing learns. The per-tile table - each search's six numbers and
every picture's score (`helpers/decide.py`) - is kept in `checkpoints/decide/`, the training
split's too, for notebook 4.

Needs: `00_data_preprocess.ipynb`. Results: `summary.md`, "Whole frames".

In [1]:
"""Settings, and the per-tile tables of both splits (cached)."""

import pickle
import time

import numpy as np

from helpers import decide, models, training

SEED, CAP = 0, 50.0
WINDOW, SHARPNESS = 15, 1.0      # notebook 1's pick, chosen on train runs
KINDS = ("page", "desktop", "photo", "screen", "video")
CACHE = training.CHECKPOINT_DIR / "decide"
PUBLISHED = "±8"

manifest = training.load_manifest()
searches = {name: decide.search(name) for name in decide.SEARCHES}
METHODS = {"tile search + pick": models.TilePick(searches[PUBLISHED], window=WINDOW, sharpness=SHARPNESS)}


def table(split):
    path = CACHE / f"{split}.pkl"
    if path.exists() and path.stat().st_mtime > (training.DATA_DIR / "manifest.json").stat().st_mtime:
        return pickle.loads(path.read_bytes())
    started = time.time()
    found = decide.load(manifest, split, searches, METHODS)
    CACHE.mkdir(parents=True, exist_ok=True)
    path.write_bytes(pickle.dumps(found))
    print(f"{split}: {len(found['moving'])} tiles in {time.time() - started:.0f}s")
    return found


tables = {split: table(split) for split in ("train", "val")}
for split, found in tables.items():
    listed = manifest["splits"][split]["sources"]
    found["kind"] = np.array([listed[s]["kind"] for s in found["source"]])
    found["group"] = np.array([listed[s].get("video", listed[s]["file"]) for s in found["source"]])
    print(f"{split:<6}{len(found['moving']):7} tiles: {found['changed'].mean():4.0%} changed, {found['moving'].mean():4.0%} moving - "
          + ", ".join(f"{k} {(found['kind'] == k).sum()}" for k in KINDS))

train   60148 tiles:  43% changed,  60% moving - page 12064, desktop 11136, photo 6960, screen 18437, video 11551
val     19374 tiles:  36% changed,  55% moving - page 3248, desktop 1392, photo 1740, screen 10652, video 2342


## Every method, by kind

Absolute wPSNR over the kept region of each tile, capped at 50 dB, per kind and over all; the gain
over the mock's mean - what the client runs today - with its 95% range over videos. The pictures
that do not move anything (hold, mean) are the same for every search.

In [2]:
found = tables["val"]
rng = np.random.default_rng(SEED)
kind, group, changed, moving = found["kind"], found["group"], found["changed"], found["moving"]
still = {"hold N": "hold N", "hold N+2": "hold N+2", "mean (the mock)": "mean"}
ROWS = {label: np.minimum(found["scores"][PUBLISHED][choice], CAP) for label, choice in still.items()}
for reach in decide.SEARCHES:
    for choice, label in (("search", "search"), ("N moved", "N moved alone"), ("N+2 moved", "N+2 moved alone")):
        ROWS[f"{label}, {reach}"] = np.minimum(found["scores"][reach][choice], CAP)
ROWS[f"tile search + pick, {PUBLISHED}"] = np.minimum(found["methods"]["tile search + pick"], CAP)


def over(gain, mask, draws=2000):
    """95% of the mean gain over `mask`, the videos (clips for drawn kinds) drawn with replacement."""
    per = [gain[mask & (group == g)] for g in np.unique(group[mask])]
    means = [np.concatenate([per[i] for i in rng.integers(0, len(per), len(per))]).mean() for _ in range(draws)]
    return np.percentile(means, [2.5, 97.5])


for title, mask in (("changed tiles", changed), ("moving tiles", moving), ("unchanged tiles", ~changed)):
    print(f"{title + ', ≤ 50 dB':<28}" + "".join(f"{k:>9}" for k in (*KINDS, "all")) + f"{'vs the mean':>13}{'95% over videos':>18}")
    print(f"{'tiles':<28}" + "".join(f"{(mask & (kind == k)).sum():9}" for k in KINDS) + f"{mask.sum():9}")
    for name, s in ROWS.items():
        gain = s - ROWS["mean (the mock)"]
        low, high = over(gain, mask) if name != "mean (the mock)" else (0.0, 0.0)
        print(f"{name:<28}" + "".join(f"{s[mask & (kind == k)].mean():9.2f}" for k in KINDS)
              + f"{s[mask].mean():9.2f}{gain[mask].mean():+13.2f}     [{low:+.2f}, {high:+.2f}]")
    print()

changed tiles, ≤ 50 dB           page  desktop    photo   screen    video      all  vs the mean   95% over videos
tiles                            1591      281      851     2440     1809     6972
hold N                          24.78    30.98    26.61    35.42    31.42    30.70        -1.46     [-2.03, -0.56]
hold N+2                        24.64    31.01    27.00    35.19    29.89    30.24        -1.92     [-2.26, -1.50]
mean (the mock)                 26.15    31.66    29.24    36.68    32.81    32.16        +0.00     [+0.00, +0.00]
search, ±8                      33.85    31.64    39.50    37.72    33.95    35.83        +3.67     [+2.53, +5.08]
N moved alone, ±8               33.14    30.67    37.68    36.62    31.97    34.51        +2.34     [+1.07, +3.84]
N+2 moved alone, ±8             33.08    30.96    38.23    36.37    31.33    34.32        +2.16     [+0.87, +3.88]
search, ±16                     35.08    31.61    39.92    37.67    34.07    36.18        +4.01     [+2.78, +5.52

## The reach, chosen on the training split

What each search costs, as `models.Interpolate` in a graph - GPU time for a 1080p frame (36 tiles
in one batch) in Chrome's WebGPU on the RTX 3090, the benchmark page's (`decide.BROWSER_MS`):

| search | offsets | GPU ms a 1080p frame |
| --- | --- | --- |
| ±8 (published) | 81 | 3.3 |
| ±16 | 289 | 8.9 |
| ±16 at ¼ | 81 | 2.9 |

±16 costs 2.7 times the published search - more than half the 16.7 ms a 60 fps frame has, on a
fast desktop GPU. So the search the later notebooks build on (`decide.reach`) is, of those no slower
than the published one, the one whose picture scores best on the training split's moving tiles -
not on validation.

In [3]:
train = tables["train"]
for name in decide.SEARCHES:
    print(f"search {name:<10}{decide.BROWSER_MS[name]:5.1f} ms in the browser, training split's moving tiles: "
          f"{np.minimum(train['scores'][name]['search'], CAP)[train['moving']].mean():.2f} dB")
REACH = decide.reach(train)
print(f"chosen: {REACH}")

search ±8          3.3 ms in the browser, training split's moving tiles: 39.56 dB
search ±16         8.9 ms in the browser, training split's moving tiles: 39.70 dB
search ±16 at ¼    2.9 ms in the browser, training split's moving tiles: 39.25 dB
chosen: ±8


## What it says

- **On whole tiles the search is +2.40 dB over the mock** on the moving tiles (40.55 against
  38.15; 95% over videos +1.59 to +3.42) and +3.67 on the changed ones (35.83 against 32.16).
- **Almost all of it is the drawn kinds.** Over the mean on the moving tiles: photos +7.5 dB, pages
  +6.8 - a pan or a scroll, one motion a tile - against +0.9 on video, +0.5 on the recorded screens
  and nothing on the drawn desktops, where a still screen holds a dragged window or a menu.
- **Holding a frame is worse than the mean** (−1.1 dB on the moving tiles), and so is moving one
  frame alone: N moved half way alone is 1.05 dB under the search. A change seen in both frames
  is better half drawn in two places than drawn whole in the wrong one, on the score.
- **The reach is worth a little, and costs a lot.** ±16 is +0.22 dB on the moving tiles - pages
  +1.06, photos +0.30, the recordings nothing - for 2.7 times the search's time in the browser;
  ±16 at a quarter of the resolution, as cheap as ±8, loses 0.54. The later notebooks keep ±8.
- **The per-pixel pick gains on whole tiles** where it lost on notebook 1's samples: +0.11 dB over
  the search on the moving tiles (notebook 4: 95% over videos +0.04 to +0.19), all of it on the
  recordings - video +0.34, screens +0.10. It costs 3.4 times the search in the browser.
- **No search harms a still tile**: on the unchanged tiles the searches and the pick are within
  0.02 dB of the mean; holding a frame, or moving one alone, loses 0.2.